&nbsp;

# Bulk RNA-seq Metadata Visualization Process

In [1]:
import plotly.graph_objects as go
import polars as pl
import pandas as pd
import plotly.colors as pc
import plotly.io as pio
import numpy as np
from typing import Any
from IPython.display import display


pio.renderers.default = "vscode"

&nbsp;

&nbsp;

# Metadata

In [2]:
rnaseq_all_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/rnaseq_all_complete/rnaseq_all_metadata_modified_all_261014.tsv",
    separator="\t",
    infer_schema_length=100000
).with_columns(
    pl.lit("heat-stress vs control pair").alias("pair_type")
).with_columns(
    (
     pl.col("stress treatment temperature") + ", " + 
     pl.col("stress treatment time")
     ).alias("study_with_conditions")
)

pl.Config.set_tbl_cols(200)
display(rnaseq_all_metadata)

sample_heat_id,sample_control_id,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,rna_library,normalized_label,item_type,strandness,strandness_evidence,expected_format_heat,expected_format_control,strand_mapping_bias_heat,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample,url,Biosample term id,Biosample term name,other term id,genotype_normalization,genotype_class,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,sample_name,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider,pair_type,study_with_conditions
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str
"""SRR24973772""","""SRR24973776""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl…"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## (IMPORTANT!) Count Information

In [3]:
project_count = rnaseq_all_metadata.select(
    pl.col("study_accession")
).to_series().n_unique()

sample_count = rnaseq_all_metadata.select(
    pl.col("Biosample term name")
).to_series().n_unique()

pair_count = rnaseq_all_metadata.select(
    pl.col("sample_heat_id"),
    pl.col("sample_control_id")
).sort(by=["sample_heat_id", "sample_control_id"]).n_unique()


print(f"Unique Curated Project Count: {project_count}")
print(f"Unique Sample Count: {sample_count}")
print(f"Unique Pair Count: {pair_count}")

Unique Curated Project Count: 53
Unique Sample Count: 41
Unique Pair Count: 406


In [4]:
sample_count_data = rnaseq_all_metadata.group_by(
    pl.col("Biosample term name"),
    pl.col("Biosample term id")
).agg(
    pl.col("sample_heat_id").n_unique().alias("sample_pair_count"),
    pl.col("study_accession").unique().alias("project"),
    pl.col("study_accession").n_unique().alias("project_count"),
).with_columns(
    # Join list[str] into one comma-separated string for TSV export.
    pl.col("project").list.sort().list.join(", ").alias("project"),
).sort(
    by=["sample_pair_count"],
    descending=True
)

pl.Config.set_tbl_rows(100)
print(sample_count_data.select("Biosample term id").unique())

pl.Config.set_tbl_rows(10)
display(sample_count_data)

shape: (41, 1)
┌───────────────────┐
│ Biosample term id │
│ ---               │
│ str               │
╞═══════════════════╡
│ CL_0000047        │
│ CLO_0051568       │
│ CLO_0037203       │
│ CLO_0007634       │
│ CL_0002540        │
│ CLO_0003665       │
│ BTO_0005250       │
│ CLO_0003684       │
│ BTO_0000552       │
│ EFO_0005907       │
│ CLO_0007865       │
│ CLO_0002785       │
│ NCIT_C172259      │
│ EFO_0007598       │
│ EFO_0022442       │
│ CL_0000312        │
│ CLO_0037372       │
│ UBERON_0010996    │
│ BTO_0004169       │
│ CLO_0003704       │
│ CLO_0002993       │
│ CLO_0001601       │
│ EFO_0002451       │
│ CLO_0037280       │
│ EFO_0005444       │
│ CL_0000359        │
│ EFO_0002067       │
│ CL_0000746        │
│ CL_0000540        │
│ CLO_0008276       │
│ CL_0000134        │
│ CLO_0009454       │
│ CLO_0007043       │
│ CLO_0009591       │
│ CLO_0007938       │
│ CLO_0050999       │
│ CLO_0001230       │
│ EFO_0022539       │
│ CLO_0009221       │
│ CL_0002139     

Biosample term name,Biosample term id,sample_pair_count,project,project_count
str,str,u32,str,u32
"""HCT 116 cell""","""CLO_0003665""",76,"""PRJNA1080614, PRJNA391584, PRJ…",3
"""HeLa cell""","""CLO_0003684""",56,"""PRJDB3761, PRJEB64703, PRJNA10…",8
"""U-2 OS cell""","""CLO_0009454""",39,"""PRJEB5979, PRJNA476608""",2
"""HEK 293 cell""","""CLO_0001230""",21,"""PRJNA1204088, PRJNA316856""",2
"""HEK293T cell""","""CLO_0037372""",19,"""PRJNA747271, PRJNA815310, PRJN…",3
…,…,…,…,…
"""cardiac muscle cell""","""CL_0000746""",2,"""PRJCA005446""",1
"""endothelial cell of vascular t…","""CL_0002139""",2,"""PRJCA005446""",1
"""mesenchymal stem cell""","""CL_0000134""",2,"""PRJCA005446""",1


&nbsp;

&nbsp;

&nbsp;

### Strandness

- 

In [5]:
strandness_count_all = rnaseq_all_metadata.filter(
    (pl.col("strandness") == "stranded")
    & 
    (
        (pl.col("expected_library_strandedness") == "stranded library pairs") |
        (pl.col("expected_library_strandedness") == "mixed strandedness library pairs")
    )
)

print(f"Unique Sample Pair Count: {strandness_count_all.select(['sample_heat_id', 'sample_control_id']).n_unique()}")
display(strandness_count_all)

Unique Sample Pair Count: 251


sample_heat_id,sample_control_id,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,rna_library,normalized_label,item_type,strandness,strandness_evidence,expected_format_heat,expected_format_control,strand_mapping_bias_heat,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample,url,Biosample term id,Biosample term name,other term id,genotype_normalization,genotype_class,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,sample_name,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider,pair_type,study_with_conditions
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str
"""SRR23989379""","""SRR23989390""","""SY5609_120_R3 vs SY5609_0_R3""",

In [6]:
strandness_count = rnaseq_all_metadata.filter(
    (pl.col("strandness") == "stranded")
    & 
    (
        (pl.col("expected_library_strandedness") == "stranded library pairs") |
        (pl.col("expected_library_strandedness") == "mixed strandedness library pairs")
    )
).group_by(
    "normalized_label"
).agg(
    pl.col("sample_heat_id").n_unique().alias("sample_pair_count"),
).sort(
    by=["sample_pair_count"],
    descending=True
)

pl.Config.set_fmt_str_lengths(100)
display(strandness_count)

normalized_label,sample_pair_count
str,u32
"""NEBNext Ultra II Directional RNA Library Prep Kit for Illumina""",87
"""TruSeq Stranded mRNA Library Prep Kit""",70
"""KAPA mRNA HyperPrep Kit""",18
"""SureSelect Strand-Specific RNA Library Preparation Kit""",17
"""MSR-seq (custom mRNA library preparation)""",12
…,…
"""TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)""",4
"""KAPA Stranded mRNA-Seq Kit""",3
"""KAPA RNA HyperPrep Kit""",3


&nbsp;

&nbsp;

&nbsp;

## Parallel Object

In [7]:
def create_parcats_from_polars(df, dimensions, color_column=None, title="Parallel Categories Diagram", width=1200, height=800):
    """
    Function to create Parallel Categories plot from Polars DataFrame.
    Uses distinct categorical colors to avoid overlap.
    """
    
    # 1. Prepare dimensions
    parcats_dimensions = []
    for col in dimensions:
        parcats_dimensions.append(dict[str, Any](
            label=col,
            values=df[col].to_list()
        ))
        
    # 2. Configure line styling
    line_props = dict[str, Any](
        color='gray',
        shape='hspline'
    )
    
    if color_column:
        # Get unique values
        unique_vals = df[color_column].unique().sort().to_list()
        n_categories = len(unique_vals)
        
        # Map categories to integer indices
        val2idx = {v: i for i, v in enumerate[Any](unique_vals)}
        color_vals = [val2idx[v] for v in df[color_column].to_list()]
        
        # --- 色の生成ロジック変更点 ---
        # 多数のカテゴリに対応するため、複数のパレットを結合して大きなパレットを作成します
        # Alphabet(26色) + Dark24(24色) + Light24(24色) ...
        distinct_palette = pc.qualitative.Alphabet + pc.qualitative.Dark24 + pc.qualitative.Light24
        
        # カテゴリ数がパレットより多い場合は繰り返す
        if n_categories > len(distinct_palette):
            distinct_palette = distinct_palette * (n_categories // len(distinct_palette) + 1)
        
        # 必要な数だけ切り出す
        custom_colorscale = distinct_palette[:n_categories]

        line_props: dict[str, Any] = dict[str, Any]()
        
        line_props.update(dict[str, Any](
            color=color_vals,
            colorscale=custom_colorscale, # カスタムパレットを適用
            cmin=0,
            cmax=n_categories - 1,
            showscale=False,
            shape='hspline'
        ))

    # 3. Create the Figure
    fig = go.Figure(data=[go.Parcats(
        dimensions=parcats_dimensions,
        line=line_props,
        hoveron='color',
        hoverinfo='count+probability',
        arrangement='freeform',
        bundlecolors=True
    )])
    
    fig.update_layout(
        title_text=title,
        font_size=18,
        width=width,
        height=height,
        margin=dict[str, int](l=80, r=80, t=80, b=80)
    )
    
    return fig

In [8]:
print(rnaseq_all_metadata.columns)

['sample_heat_id', 'sample_control_id', 'sample_title', 'experiment_accession', 'experiment_alias', 'secondary_sample_accession', 'study_accession', 'first_public', 'study_alias', 'instrument_platform', 'instrument_model', 'library_source', 'library_layout', 'sample_molecule_ch1', 'rna_library', 'normalized_label', 'item_type', 'strandness', 'strandness_evidence', 'expected_format_heat', 'expected_format_control', 'strand_mapping_bias_heat', 'strand_mapping_bias_control', 'format_pair_comparison', 'expected_library_strandedness', 'sample', 'url', 'Biosample term id', 'Biosample term name', 'other term id', 'genotype_normalization', 'genotype_class', 'stress treatment temperature', 'stress treatment time', 'control treatment temperature', 'reference_doi', 'sample_name', 'sample_geo_accession', 'sample_status', 'sample_submission_date', 'sample_last_update_date', 'sample_type', 'sample_channel_count', 'sample_source_name_ch1', 'sample_organism_ch1', 'sample_taxid_ch1', 'sample_descriptio

In [9]:
fig = create_parcats_from_polars(
    df=rnaseq_all_metadata, 
    dimensions=["study_accession", "strandness", "sample_molecule_ch1", "stress treatment temperature", "stress treatment time", "Biosample term name"], 
    color_column="study_accession",
    title="Metadata Flow",
    height=1200,
    width=1800
)

fig.show()

&nbsp;

&nbsp;

&nbsp;

## HN-score Category

- transcript level で、`UP`, `DOWN`, `UNCHANGED` も積み上げで可視化するための処理

In [10]:
HNratio_categorized = pl.read_csv(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/HNratio_categorized_2.tsv",
    separator="\t",
    n_threads=16
)

display(HNratio_categorized)

FileNotFoundError: No such file or directory (os error 2): ../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/HNratio_categorized_2.tsv

In [ ]:
# Create a dictionary for renaming columns, removing the '_category' suffix from all columns except 'GENEID'.
rename_mapping = {
    col: col.removesuffix('_category') 
    for col in HNratio_categorized.columns 
    if col != 'transcript_id'
}

# Chain the rename and transpose operations.
# 'transpose' makes rows into columns and vice-versa.
# 'include_header=True' keeps the original column names as a new column.
# 'header_name' gives a name to that new column.
# 'column_names' uses values from this column as the new headers.
HNratio_transposed = HNratio_categorized.rename(rename_mapping).transpose(
    include_header=True, 
    header_name="sample_heat_id", 
    column_names="transcript_id"
)

display(HNratio_transposed)

In [ ]:
rnaseq_all_metadata_join = rnaseq_all_metadata.join(
    HNratio_transposed,
    on="sample_heat_id",
    how="inner"
)

display(rnaseq_all_metadata_join)

&nbsp;

&nbsp;

## Case Study 1: _NEAT1_

In [ ]:
# NEAT1 の transcript の一覧を取得
HNscore_NEAT1 = pl.read_parquet(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HNscore_gencode_human_261001_HN2_transcript_all_gencode_annotation.parquet",
).filter(
    pl.col("gene_id").str.contains("ENSG00000245532")
)

pl.Config.set_tbl_rows(100)
display(HNscore_NEAT1)

In [ ]:
NEAT1_transcripts = rnaseq_all_metadata_join.select(
    "sample_heat_id",
    "study_accession",
    "sample_molecule_ch1",
    "Biosample term name",
    "stress treatment temperature",
    "stress treatment time",
    "ENST00000813135.1",
    "ENST00000813134.1",
    "ENST00000616315.2",
    "ENST00000501122.4",
    "ENST00000642367.1",
    "ENST00000813132.1",
    "ENST00000670617.1",
    "ENST00000717968.1",
    "ENST00000612303.3",
    "ENST00000813130.1",
    "ENST00000813131.1",
    "ENST00000813133.1",
    "ENST00000601801.3",
    "ENST00000645023.1",
    "ENST00000646243.1",
    "ENST00000499732.4"
)

pl.Config.set_tbl_rows(2)
display(NEAT1_transcripts)

In [ ]:
DRR321029 = NEAT1_transcripts.filter(
    pl.col("sample_heat_id").str.contains("DRR321029")
)

display(DRR321029)

In [ ]:
fig = create_parcats_from_polars(
    df=NEAT1_transcripts, 
    dimensions=["ENST00000501122.4", "Biosample term name", "sample_molecule_ch1"], 
    color_column="study_accession",
    title="Metadata Flow",
    height=1200,
    width=1800
)

fig.show()

&nbsp;

&nbsp;

&nbsp;

## Case Study 2: 